In [38]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

In [24]:
SEED: int = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Data pipeline

In [ ]:
def load_dataset(path: str) -> pd.DataFrame:
    return pd.read_csv(path)

def filter_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Removes network-specific socket features and junk columns
    to prevent model overfitting.
    """
    df.columns = df.columns.str.strip()
    columns_to_drop: list[str] = [
        'Unnamed: 0',
        'Flow ID',
        'Source IP',
        'Source Port',
        'Destination IP',
        'Destination Port',
        'Timestamp',
        'Fwd Header Length.1',
        'SimillarHTTP',
        'Inbound'
    ]
    existing_columns_to_drop: list[str] = [col for col in columns_to_drop if col in df.columns]
    filtered_df: pd.DataFrame = df.drop(existing_columns_to_drop, axis=1)

    return filtered_df

def clean_missing_and_infinite_values(df: pd.DataFrame) -> pd.DataFrame:
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(axis=0, how="any", inplace=True)

    return df

def encode_labels(df: pd.DataFrame) -> pd.DataFrame:
    df["Label"] = df["Label"].astype(str).str.strip().str.upper()
    df["Label"] = (df["Label"] != "BENIGN").astype(int)
    return df


In [28]:
def load_and_concat_datasets(directory_path: str, samples_per_file: int = 15_000) -> pd.DataFrame:
    processed_dfs: list[pd.DataFrame] = []
    for filename in os.listdir(directory_path):
        if filename.endswith(".csv"):
            file_path: str = os.path.join(directory_path, filename)
            temp_df: pd.DataFrame = load_dataset(file_path)
            temp_df = filter_features(temp_df)
            temp_df = clean_missing_and_infinite_values(temp_df)

            if len(temp_df) > samples_per_file:
                temp_df = temp_df.sample(n=samples_per_file, random_state=SEED)
            else:
                temp_df = temp_df.sample(frac=1.0, random_state=SEED)
            encode_labels(temp_df)
            processed_dfs.append(temp_df)

    master_df: pd.DataFrame = pd.concat(processed_dfs, ignore_index=True)
    return master_df

In [32]:
DATA_DIR: str = "data/01-12"
MASTER_DATASET_PATH: str = "data/master_clean_dataset.csv"

master_df: pd.DataFrame = load_and_concat_datasets(DATA_DIR)
master_df.to_csv(MASTER_DATASET_PATH, index=False)
print(master_df.shape)
print(master_df.head())


/var/folders/r4/23ff9xvd1w1b802q18kbr7tr0000gn/T/ipykernel_68403/3512314337.py:2: DtypeWarning: Columns (0: SimillarHTTP) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path)
/var/folders/r4/23ff9xvd1w1b802q18kbr7tr0000gn/T/ipykernel_68403/3512314337.py:2: DtypeWarning: Columns (0: SimillarHTTP) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path)
/var/folders/r4/23ff9xvd1w1b802q18kbr7tr0000gn/T/ipykernel_68403/3512314337.py:2: DtypeWarning: Columns (0: SimillarHTTP) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path)
/var/folders/r4/23ff9xvd1w1b802q18kbr7tr0000gn/T/ipykernel_68403/3512314337.py:2: DtypeWarning: Columns (0: SimillarHTTP) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path)
/var/folders/r4/23ff9xvd1w1b802q18kbr7tr0000gn/T/ipykernel_68403/3512314337.py:2: DtypeWarning: Columns (0: 

(165000, 78)
   Protocol  Flow Duration  Total Fwd Packets  Total Backward Packets  \
0        17            204                 52                       0   
1        17           1091                126                       0   
2        17           1250                200                       0   
3        17            327                 20                       0   
4        17            653                 62                       0   

   Total Length of Fwd Packets  Total Length of Bwd Packets  \
0                      22736.0                          0.0   
1                      55440.0                          0.0   
2                      88000.0                          0.0   
3                       8080.0                          0.0   
4                      27136.0                          0.0   

   Fwd Packet Length Max  Fwd Packet Length Min  Fwd Packet Length Mean  \
0                  440.0                  368.0              437.230769   
1                  

In [48]:
def split_dataframe(df: pd.DataFrame, train_frac: float = 0.7) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    X: np.ndarray = df.drop("Label", axis=1).values
    y: np.ndarray = df["Label"].values

    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=(1.0-train_frac), random_state=SEED, stratify=y
    )

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
    )

    return X_train, y_train, X_val, y_val, X_test, y_test

def normalize(X_train: np.ndarray, X_val: np.ndarray, X_test: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    scaler: MinMaxScaler = MinMaxScaler(feature_range=(0,1))
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_val_scaled, X_test_scaled

X_train, y_train, X_val, y_val, X_test, y_test = split_dataframe(master_df)
X_train, X_val, X_test = normalize(X_train, X_val, X_test)



In [49]:
class DDOSDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).long()

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, index: int):
        return self.X[index], self.y[index]

In [50]:
train_dataset = DDOSDataset(X_train, y_train)
valid_dataset = DDOSDataset(X_val, y_val)
test_dataset = DDOSDataset(X_test, y_test)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)